# Telugu Handwritten Character Recognizer (v4)
### Multi-Head EfficientNetV2 with Constrained Maximum-Likelihood Decoding

This notebook runs the complete training and evaluation pipeline on Kaggle GPU (T4 / P100 / A100).

In [ ]:
# 1. Verify GPU and Environment
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs available:', gpus)
if gpus:
    print('Using GPU:', gpus[0])
else:
    print('WARNING: No GPU detected!')

In [ ]:
# 2. Mount Dataset Paths & Generate / Remap Frozen Splits
import os
from pathlib import Path

# Locate dataset directory on Kaggle
possible_paths = [
    Path('/kaggle/input/telugu-handwritten-character-dataset/Final Dataset of Telugu Handwritten Chararcters/Test1'),
    Path('/kaggle/input/telugu-hcr/Final Dataset of Telugu Handwritten Chararcters/Test1'),
    Path('/kaggle/input/telugu-dataset/Test1'),
    Path('data/Final Dataset of Telugu Handwritten Chararcters/Test1'),
]

dataset_root = None
for p in possible_paths:
    if p.exists():
        dataset_root = p
        break

print('Detected dataset root:', dataset_root)

# If outputs/train.csv does not exist, generate split
if not Path('outputs/train.csv').exists() and dataset_root:
    from src.data.split import create_frozen_splits
    create_frozen_splits(dataset_root=str(dataset_root), output_dir='outputs', seed=42)
else:
    print('Frozen splits ready in outputs/')

In [ ]:
# 3. Run Training with Two-Phase Warmup + Fine-Tuning
from src.train import run_training

# Launch training (default EfficientNetV2B0, batch size 128)
run_training(
    config_path='configs/multitask_effnetv2.yaml',
    resume=False,
    custom_epochs=45,
    custom_batch_size=128,
    custom_variant='B0'
)

In [ ]:
# 4. Rigorous Evaluation on Test Set
from src.evaluate import evaluate_test_set

report = evaluate_test_set(
    test_csv='outputs/test.csv',
    label_maps_path='outputs/label_maps.json',
    checkpoint_dir='checkpoints',
    checkpoint_tag='best_model',
    variant='B0',
    batch_size=128,
    output_report_path='outputs/evaluation_report.json'
)

import json
print(json.dumps(report, indent=2))